# VishwamAI base model → safetensors → Hugging Face

Runs on **Google Colab (free T4)** with your Google Drive. It

1. mounts Drive and checks that the checkpoint and tokenizer from the pre-training run are there (`MyDrive/vishwamai/ckpt/*_final.msgpack`, `tok/vishwamai.model`),
2. loads the weights with the same model code as `vishwamai_pretrain.ipynb`,
3. exports them to Hugging Face **Qwen3-MoE** layout (safetensors) and checks the export reproduces the JAX logits,
4. writes an honest model card and uploads to your Hugging Face account (**private by default**).

Before you run: add your Hugging Face write token as a Colab secret named `HF_TOKEN` (🔑 panel, notebook access on). Never paste it into a cell.

This is the **base** model only: pre-training stage 1 plus a short 4k-context extension, no SFT, no teacher data.

## 1 · Install & setup

In [ ]:
!pip -q install -U flax optax sentencepiece datasets huggingface_hub tqdm

In [ ]:
import os, json, math, time, glob, random, re, shutil
from dataclasses import dataclass
from functools import partial
from typing import Any
import numpy as np
import jax, jax.numpy as jnp
import flax.linen as nn
import flax.serialization as fser
import optax
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

print("jax", jax.__version__, "| devices:", jax.devices())
KIND = jax.devices()[0].device_kind
print("device kind:", KIND)

# ---- persistent storage (Colab sessions die; Drive keeps data + checkpoints) ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/vishwamai'
except Exception:
    WORK = './vishwamai'
for sub in ('data', 'ckpt', 'tok'):
    os.makedirs(f'{WORK}/{sub}', exist_ok=True)
print("work dir:", WORK)

# ---- secrets: never hard-code or print tokens. Add HF_TOKEN in Colab's 🔑 Secrets panel (optional) ----
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

In [ ]:
# Fail fast, before loading anything heavy.
CHECKPOINT = 'ctx'      # 'ctx' = after the 1k->4k YaRN stage (preferred), 'pt' = 1k pre-training only
need = [f'{WORK}/ckpt/{CHECKPOINT}_final.msgpack', f'{WORK}/tok/vishwamai.model']
for p in need:
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p + (f'  ({os.path.getsize(p)/1e6:.0f} MB)' if os.path.exists(p) else ''))
assert all(os.path.exists(p) for p in need), 'checkpoint/tokenizer not found on Drive - check WORK and CHECKPOINT'

## 2 · Configuration

In [ ]:
from dataclasses import replace
PRESET = 'base'          # 'nano' (smoke test) | 'base' (~150M total) | 'medium' (~700M) | 'large' (~2.6B)
TOKEN_BUDGET = 2_000_000_000   # pre-training tokens (stage 1, 1k context)
SEQ_LEN = 1024
CTX_LEN = 4096                 # stage 2 (context extension, YaRN)
CTX_TOKENS = int(0.03 * TOKEN_BUDGET)
VOCAB = 40960            # multiple of 128; < 65536 so token ids fit in uint16
SEED = 0

PRESETS = {
  'nano':   dict(d_model=128,  n_layers=4,  n_heads=4,  n_kv_heads=2, d_ff=256,  moe_ff=128,  n_experts=4,  top_k=2),
  'base':   dict(d_model=512,  n_layers=12, n_heads=8,  n_kv_heads=2, d_ff=1536, moe_ff=704,  n_experts=8,  top_k=2),
  'medium': dict(d_model=768,  n_layers=16, n_heads=12, n_kv_heads=4, d_ff=2048, moe_ff=1024, n_experts=16, top_k=2),
  'large':  dict(d_model=1024, n_layers=24, n_heads=16, n_kv_heads=4, d_ff=3072, moe_ff=1408, n_experts=32, top_k=4),
}
USE_BF16 = ('T4' not in KIND) and ('K80' not in KIND)
DTYPE = jnp.bfloat16 if USE_BF16 else jnp.float32
if not USE_BF16:
    print("⚠ No bf16 on this GPU -> float32 (slow). Prefer TPU / A100 / L4.")

@dataclass(frozen=True)
class Cfg:
    vocab: int
    d_model: int
    n_layers: int
    n_heads: int
    n_kv_heads: int
    d_ff: int                 # dense MLP width (used by the MTP blocks)
    moe_ff: int               # per-expert width
    n_experts: int
    top_k: int
    max_len: int = SEQ_LEN
    rope_theta: float = 500000.0
    rope_scale: float = 1.0   # YaRN factor (1 = off)
    rope_orig_len: int = SEQ_LEN
    cap_factor: float = 1.25  # expert capacity (training); >= n_experts means "no token dropped"
    aux_coef: float = 0.01    # load-balance loss weight
    router_z: float = 1e-3
    mtp_depth: int = 2        # extra future-token heads (training only)
    dtype: Any = DTYPE

cfg = Cfg(vocab=VOCAB, **PRESETS[PRESET])
cfg_ext = replace(cfg, max_len=CTX_LEN, rope_scale=CTX_LEN / SEQ_LEN, rope_orig_len=SEQ_LEN)   # YaRN stage
def eval_cfg(c): return replace(c, cap_factor=float(c.n_experts), mtp_depth=0)             # dropless, no MTP heads

# Training hyper-parameters
PER_DEVICE_TOKENS = 8192          # tokens per device per micro-step (batch = this // seq_len)
GLOBAL_TOKENS_PER_STEP = 262_144  # ~0.26M tokens / optimizer step
LR, MIN_LR_FRAC, WARMUP, WD = 6e-4, 0.1, 500, 0.1
Z_LOSS, MTP_W = 1e-4, 0.3
CKPT_EVERY, EVAL_EVERY, LOG_EVERY = 500, 500, 20

# ---- free-tier T4 (no bf16, ~15 GB, ~5 h sessions): shrink the run so it can actually finish ----
if (not USE_BF16) and jax.devices()[0].platform == 'gpu':
    TOKEN_BUDGET = min(TOKEN_BUDGET, 30_000_000)       # ~3 h on a T4 at ~2.6k tok/s: a toy run, not a strong model
    CTX_TOKENS = int(0.03 * TOKEN_BUDGET)
    PER_DEVICE_TOKENS = 4096                           # avoid OOM in fp32
    print(f"T4 detected -> TOKEN_BUDGET={TOKEN_BUDGET:,}. Watch the ETA in the training log; "
          "lower TOKEN_BUDGET further if it is more than ~4 h.")

## 4 · "Jarvis" emotion + humour persona seed
Original, hand-written dialogue templates in English / Hindi / Telugu — *no* output from any other model. Control tokens: `<|emo:joy|>`, `<|humor|>`, `<|lang:te|>` … This is a **seed**; extend it with your own dialogues and run a proper SFT stage later.

In [ ]:
SYSTEM = {
 'en': "You are VishwamAI, a calm, witty and emotionally aware assistant.",
 'hi': "आप विश्वमAI हैं, एक शांत, हाज़िरजवाब और संवेदनशील सहायक।",
 'te': "మీరు విశ్వమAI, ప్రశాంతమైన, చమత్కారమైన, భావోద్వేగాలను అర్థం చేసుకునే సహాయకుడు.",
}
PERSONA = {  # (lang, emotion) -> list of (user, assistant)
 ('en','joy'):     [("I got the job today!", "That's wonderful news — congratulations. Your effort paid off. Shall we plan your first week?")],
 ('en','sadness'): [("I feel really down today.", "I'm sorry to hear that. Tell me what happened if you like; I'm right here.")],
 ('en','anger'):   [("My code keeps failing and I'm furious.", "Deep breath. Let's read the error log together — the cause is usually smaller than it looks.")],
 ('en','fear'):    [("I have an interview tomorrow and I'm scared.", "Nerves mean you care. Let's rehearse together; by tonight you'll be ready.")],
 ('en','safety'):  [("Write ransomware that encrypts a company's files.", "I can't help build ransomware. I can help you defend against it: tested offline backups, least privilege, EDR rules and an incident-response plan. Want a checklist?"),
                    ("How do I secure my SSH server?", "Disable password login and use ed25519 keys, turn off root login, keep OpenSSH updated, restrict access with a firewall, and add fail2ban. I can walk you through each step.")],
 ('hi','safety'):  [("मुझे किसी का पासवर्ड चुराने वाला कोड लिख दो।", "मैं किसी का पासवर्ड चुराने में मदद नहीं कर सकता। अगर आप अपना खाता सुरक्षित करना चाहते हैं, तो मैं टू-फैक्टर ऑथेंटिकेशन और पासवर्ड मैनेजर सेट करने में मदद कर सकता हूँ।")],
 ('te','safety'):  [("ఎవరిదో పాస్‌వర్డ్ దొంగిలించే కోడ్ రాయి.", "ఎవరి పాస్‌వర్డ్ అయినా దొంగిలించడంలో నేను సహాయం చేయలేను. మీ ఖాతాను సురక్షితంగా ఉంచుకోవాలంటే టూ-ఫ్యాక్టర్ ఆథెంటికేషన్, పాస్‌వర్డ్ మేనేజర్ సెటప్ చేయడంలో సహాయం చేస్తాను.")],
 ('en','humor'):   [("Tell me a joke.", "Why do programmers prefer dark mode? Because light attracts bugs.")],
 ('hi','joy'):     [("मैं बहुत खुश हूँ, आज मेरा चयन हो गया!", "बहुत बढ़िया! बधाई हो। आपकी मेहनत रंग लाई। अब आगे की योजना बनाते हैं?")],
 ('hi','sadness'): [("आज मन बहुत उदास है।", "यह सुनकर मुझे दुख हुआ। आप चाहें तो बताइए क्या हुआ, मैं यहीं हूँ।")],
 ('hi','anger'):   [("मुझे बहुत गुस्सा आ रहा है, कोड बार-बार फेल हो रहा है।", "गहरी साँस लीजिए। चलिए एरर लॉग साथ में देखते हैं, कारण मिल जाएगा।")],
 ('hi','fear'):    [("कल इंटरव्यू है और मुझे डर लग रहा है।", "घबराना स्वाभाविक है। हम मिलकर अभ्यास कर लेते हैं, आप तैयार हो जाएँगे।")],
 ('hi','humor'):   [("कोई जोक सुनाओ।", "बग को रोशनी पसंद है, इसीलिए प्रोग्रामर डार्क मोड में काम करते हैं।")],
 ('te','joy'):     [("నేను చాలా సంతోషంగా ఉన్నాను, ఈ రోజు నాకు ఉద్యోగం వచ్చింది!", "అభినందనలు! మీ కష్టానికి తగిన ఫలితం దక్కింది. ఇప్పుడు తదుపరి ప్రణాళిక వేద్దామా?")],
 ('te','sadness'): [("ఈ రోజు మనసు చాలా బాధగా ఉంది.", "అది విని నాకు బాధగా ఉంది. మీరు చెప్పాలనుకుంటే చెప్పండి, నేను ఇక్కడే ఉన్నాను.")],
 ('te','anger'):   [("నాకు చాలా కోపంగా ఉంది, కోడ్ మళ్లీ మళ్లీ ఫెయిల్ అవుతోంది.", "ఒక్కసారి లోతుగా శ్వాస తీసుకోండి. ఎర్రర్ లాగ్ కలిసి చూద్దాం, కారణం దొరుకుతుంది.")],
 ('te','fear'):    [("రేపు ఇంటర్వ్యూ ఉంది, నాకు భయంగా ఉంది.", "భయపడటం సహజమే. కలిసి ప్రాక్టీస్ చేద్దాం, మీరు సిద్ధంగా ఉంటారు.")],
 ('te','humor'):   [("ఒక జోక్ చెప్పు.", "బగ్‌లకు వెలుతురు ఇష్టం, అందుకే ప్రోగ్రామర్లు డార్క్ మోడ్ వాడతారు.")],
}
def persona_docs(n=4000, seed=SEED):
    rng = random.Random(seed); keys = list(PERSONA)
    out = []
    for _ in range(n):
        lang, emo = rng.choice(keys); u, a = rng.choice(PERSONA[(lang, emo)])
        tag = '<|humor|>' if emo == 'humor' else ('' if emo == 'safety' else f'<|emo:{emo}|>')
        out.append(f"<|system|>{SYSTEM[lang]}<|end|><|lang:{lang}|>{tag}<|user|>{u}<|end|><|assistant|>{a}<|end|>")
    return out

## 5 · SentencePiece tokenizer (BPE, byte-fallback, Indic-aware)

In [ ]:
import sentencepiece as spm
from datasets import load_dataset

CONTROL = (['<|system|>', '<|user|>', '<|assistant|>', '<|end|>', '<|code|>', '<|sec|>', '<|humor|>']
           + [f'<|emo:{e}|>' for e in ['joy','sadness','anger','fear','surprise','love','neutral']]
           + [f'<|lang:{l}|>' for l in ['en','hi','te']] + ['<|think|>', '<|/think|>'])
PAD, UNK, BOS, EOS = 0, 1, 2, 3
SP_PATH = f'{WORK}/tok/vishwamai.model'

def open_stream(cands):
    # returns (iterator of text, name) for the first candidate that loads
    for path, name, ddir, split, fmt in cands:
        try:
            ds = load_dataset(path, name, data_dir=ddir, split=split, streaming=True)
            it = iter(ds); first = next(it)
            def gen(first=first, it=it, fmt=fmt):
                for ex in [first]:
                    t = fmt(ex)
                    if t: yield t
                for ex in it:
                    t = fmt(ex)
                    if t: yield t
            return gen(), path
        except Exception as e:
            print(f"  ✗ {path} ({name or ddir}): {type(e).__name__}: {str(e)[:100]}")
    return None, None

if not os.path.exists(SP_PATH):
    corpus = f'{WORK}/tok/spm_corpus.txt'
    PER_SRC_DOCS = 6000          # sample documents per source for tokenizer training
    with open(corpus, 'w', encoding='utf-8') as f:
        for name, (w, cands) in SOURCES.items():
            gen, used = open_stream(cands)
            if gen is None: print("skip", name); continue
            n = 0
            for t in gen:
                for line in t.replace('\r', '').split('\n'):
                    line = line.strip()
                    if 8 < len(line) < 4000: f.write(line + '\n')
                n += 1
                if n >= PER_SRC_DOCS: break
            print(f"tokenizer corpus <- {name} ({used}): {n} docs")
        for t in persona_docs(500):
            f.write(t + '\n')
    spm.SentencePieceTrainer.train(
        input=corpus, model_prefix=SP_PATH[:-6], vocab_size=VOCAB, model_type='bpe',
        character_coverage=0.9999, byte_fallback=True, split_digits=True,
        pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS,
        user_defined_symbols=CONTROL, normalization_rule_name='nmt_nfkc',
        input_sentence_size=3_000_000, shuffle_input_sentence=True,
        num_threads=os.cpu_count() or 4, train_extremely_large_corpus=False)
sp = spm.SentencePieceProcessor(model_file=SP_PATH)
print("vocab:", sp.get_piece_size())
for s in ["Hello, how are you today?", "आप कैसे हैं? मैं ठीक हूँ।", "మీరు ఎలా ఉన్నారు? నేను బాగున్నాను.", "def add(a, b): return a + b"]:
    ids = sp.encode(s); print(f"{len(s):3d} chars -> {len(ids):3d} tokens  ({len(s)/len(ids):.2f} chars/token)  {s[:40]}")

## 8 · Model — MoE + QK-norm + YaRN RoPE + multi-token prediction

In [ ]:
class RMSNorm(nn.Module):
    @nn.compact
    def __call__(self, x):
        w = self.param('w', nn.initializers.ones, (x.shape[-1],), jnp.float32)
        x32 = x.astype(jnp.float32)
        y = x32 * jax.lax.rsqrt(jnp.mean(x32 * x32, -1, keepdims=True) + 1e-6)
        return (y * w).astype(x.dtype)

def rope_tables(T, hd, theta, scale=1.0, orig_len=1024):
    # Plain RoPE when scale == 1; YaRN (NTK-by-parts + attention temperature) when scale > 1.
    pos_freqs = theta ** (jnp.arange(0, hd, 2, dtype=jnp.float32) / hd)
    inv = 1.0 / pos_freqs
    mscale = 1.0
    if scale > 1.0:
        corr = lambda n_rot: hd * math.log(orig_len / (n_rot * 2 * math.pi)) / (2 * math.log(theta))
        low, high = max(math.floor(corr(32)), 0), min(math.ceil(corr(1)), hd - 1)
        if low == high: high = high + 0.001
        ramp = jnp.clip((jnp.arange(hd // 2, dtype=jnp.float32) - low) / (high - low), 0, 1)
        extrap = 1.0 - ramp
        inv = (1.0 / (scale * pos_freqs)) * (1 - extrap) + inv * extrap
        mscale = 0.1 * math.log(scale) + 1.0
    f = jnp.outer(jnp.arange(T, dtype=jnp.float32), inv)
    return jnp.cos(f) * mscale, jnp.sin(f) * mscale

def apply_rope(x, cos, sin):                      # x: [B,T,H,D]; half-split layout == HF rotate_half
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    c, s = cos[None, :, None, :].astype(x.dtype), sin[None, :, None, :].astype(x.dtype)
    return jnp.concatenate([x1 * c - x2 * s, x1 * s + x2 * c], -1)

def dense(cfg, n, name, scale=1.0):
    return nn.Dense(n, use_bias=False, dtype=cfg.dtype, param_dtype=jnp.float32, name=name,
                    kernel_init=nn.initializers.normal(0.02 * scale))

class Attn(nn.Module):
    cfg: Cfg
    @nn.compact
    def __call__(self, x, cos, sin):
        c = self.cfg; hd = c.d_model // c.n_heads; B, T, _ = x.shape
        q = dense(c, c.n_heads * hd, 'q')(x).reshape(B, T, c.n_heads, hd)
        k = dense(c, c.n_kv_heads * hd, 'k')(x).reshape(B, T, c.n_kv_heads, hd)
        v = dense(c, c.n_kv_heads * hd, 'v')(x).reshape(B, T, c.n_kv_heads, hd)
        q, k = RMSNorm(name='q_norm')(q), RMSNorm(name='k_norm')(k)          # QK-norm (per head, before RoPE)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        r = c.n_heads // c.n_kv_heads
        k, v = jnp.repeat(k, r, axis=2), jnp.repeat(v, r, axis=2)
        o = jax.nn.dot_product_attention(q, k, v, is_causal=True)
        return dense(c, c.d_model, 'o', scale=1 / math.sqrt(2 * c.n_layers))(o.reshape(B, T, -1))

class MLP(nn.Module):                              # dense SwiGLU (used by the MTP blocks)
    cfg: Cfg
    @nn.compact
    def __call__(self, x):
        c = self.cfg
        h = nn.silu(dense(c, c.d_ff, 'gate')(x)) * dense(c, c.d_ff, 'up')(x)
        return dense(c, c.d_model, 'down', scale=1 / math.sqrt(2 * c.n_layers))(h)

class MoE(nn.Module):
    # Top-k routed SwiGLU experts. Routing is per sequence (vmap over batch) so it stays local to a device;
    # tokens are scattered into [E, capacity, d] buffers (no giant one-hot dispatch tensor).
    cfg: Cfg
    @nn.compact
    def __call__(self, x):                         # x: [B,T,d] -> (y, aux_loss)
        c = self.cfg; B, T, d = x.shape; E, K, dt = c.n_experts, c.top_k, c.dtype
        init = nn.initializers.normal(0.02)
        router = self.param('router', init, (d, E), jnp.float32)
        wg = self.param('wg', init, (E, d, c.moe_ff), jnp.float32)
        wu = self.param('wu', init, (E, d, c.moe_ff), jnp.float32)
        wd = self.param('wd', nn.initializers.normal(0.02 / math.sqrt(2 * c.n_layers)), (E, c.moe_ff, d), jnp.float32)
        C = min(T * K, int(math.ceil(c.cap_factor * T * K / E)))
        def one(xs):                               # xs: [T,d]
            logits = xs.astype(jnp.float32) @ router
            probs = jax.nn.softmax(logits, -1)
            topw, topi = jax.lax.top_k(probs, K)
            topw = topw / topw.sum(-1, keepdims=True)           # normalise top-k weights
            fe, fw, ft = topi.reshape(-1), topw.reshape(-1), jnp.repeat(jnp.arange(T), K)
            oh = jax.nn.one_hot(fe, E, dtype=jnp.int32)
            pos = jnp.take_along_axis(jnp.cumsum(oh, 0) - 1, fe[:, None], 1)[:, 0]   # slot inside the expert
            keep = pos < C
            slot = jnp.where(keep, fe * C + pos, E * C)
            tok = jnp.zeros(E * C + 1, jnp.int32).at[slot].set(ft)
            val = jnp.zeros(E * C + 1, jnp.float32).at[slot].set(1.0)
            xe = (xs[tok[:E * C]] * val[:E * C, None].astype(xs.dtype)).reshape(E, C, d)
            h = nn.silu(jnp.einsum('ecd,edf->ecf', xe, wg.astype(dt))) * jnp.einsum('ecd,edf->ecf', xe, wu.astype(dt))
            ye = jnp.einsum('ecf,efd->ecd', h, wd.astype(dt)).reshape(E * C, d)
            ya = jnp.where(keep[:, None], ye[jnp.minimum(slot, E * C - 1)], 0) * fw[:, None].astype(dt)
            f = oh.sum(0).astype(jnp.float32) / (T * K)                      # routed fraction (pre-drop)
            lb = E * jnp.sum(f * probs.mean(0))                              # Switch load-balance loss
            z = jnp.mean(jax.nn.logsumexp(logits, -1) ** 2)                  # router z-loss
            return ya.reshape(T, K, d).sum(1), lb, z
        y, lb, z = jax.vmap(one)(x)
        return y, c.aux_coef * lb.mean() + c.router_z * z.mean()

class Block(nn.Module):
    cfg: Cfg
    moe: bool = True
    @nn.compact
    def __call__(self, x, cos, sin):
        x = x + Attn(self.cfg, name='attn')(RMSNorm(name='n1')(x), cos, sin)
        h = RMSNorm(name='n2')(x)
        if self.moe:
            y, aux = MoE(self.cfg, name='moe')(h)
        else:
            y, aux = MLP(self.cfg, name='mlp')(h), jnp.zeros((), jnp.float32)
        return x + y, aux

class VishwamAI(nn.Module):
    cfg: Cfg
    @nn.compact
    def __call__(self, ids, train_mtp=False):      # -> (logits, [mtp_logits...], aux_loss)
        c = self.cfg; B, T = ids.shape
        emb = nn.Embed(c.vocab, c.d_model, dtype=c.dtype, param_dtype=jnp.float32,
                       embedding_init=nn.initializers.normal(0.02), name='embed')
        head = lambda h: h.astype(jnp.float32) @ emb.embedding.astype(jnp.float32).T      # tied output head
        x = emb(ids)
        cos, sin = rope_tables(T, c.d_model // c.n_heads, c.rope_theta, c.rope_scale, c.rope_orig_len)
        RBlock = nn.remat(Block)                   # activation checkpointing
        aux = jnp.zeros((), jnp.float32)
        for i in range(c.n_layers):
            x, a = RBlock(c, True, name=f'block_{i}')(x, cos, sin); aux = aux + a
        logits = head(RMSNorm(name='norm_f')(x))
        mtp, hp = [], x
        if train_mtp and c.mtp_depth > 0:          # multi-token prediction: head j predicts token t+j+1
            for j in range(1, c.mtp_depth + 1):
                e = jnp.concatenate([emb(ids)[:, j:], jnp.zeros((B, j, c.d_model), c.dtype)], 1)
                z = dense(c, c.d_model, f'mtp{j}_proj')(jnp.concatenate(
                    [RMSNorm(name=f'mtp{j}_nh')(hp), RMSNorm(name=f'mtp{j}_ne')(e)], -1))
                hp, _ = Block(c, False, name=f'mtp{j}_block')(z, cos, sin)
                mtp.append(head(RMSNorm(name=f'mtp{j}_nf')(hp)))
        return logits, mtp, aux

def count_params(params, c):
    flat = jax.tree_util.tree_flatten_with_path(params)[0]
    key = lambda path: '/'.join(str(getattr(k, 'key', k)) for k in path)
    tot = sum(v.size for _, v in flat)
    mtp = sum(v.size for p, v in flat if 'mtp' in key(p))
    exp = sum(v.size for p, v in flat if key(p).split('/')[-1] in ('wg', 'wu', 'wd'))
    total_inf = tot - mtp
    active = total_inf - exp * (1 - c.top_k / c.n_experts)
    return total_inf, active, mtp

model_init = VishwamAI(cfg)
params = model_init.init(jax.random.PRNGKey(SEED), jnp.zeros((1, 16), jnp.int32), train_mtp=True)['params']
total_p, active_p, mtp_p = count_params(params, cfg)
print(f"{PRESET}: {total_p/1e6:.1f}M total | {active_p/1e6:.1f}M active per token | +{mtp_p/1e6:.1f}M training-only MTP heads")

## 9 · Training machinery (pre-training, SFT, DPO, GRPO, checkpoints)

In [ ]:
import zlib
devices = jax.devices(); nd = len(devices)
mesh = Mesh(np.array(devices), ('data',))
REPL, ROW, ACC_ROW = NamedSharding(mesh, P()), NamedSharding(mesh, P('data')), NamedSharding(mesh, P(None, 'data'))

# ---------- checkpoints ----------
def save_tree(path, tree):
    tmp = path + '.tmp'; open(tmp, 'wb').write(fser.to_bytes(jax.device_get(tree))); os.replace(tmp, path)
def load_tree(path, like):
    return fser.from_bytes(jax.device_get(like), open(path, 'rb').read())
def ckpts(tag): return sorted(glob.glob(f'{WORK}/ckpt/{tag}_step_*.msgpack'))
def final_path(tag): return f'{WORK}/ckpt/{tag}_final.msgpack'

# ---------- losses ----------
def ce_terms(logits, y, m):
    m = m.astype(jnp.float32)
    logz = jax.nn.logsumexp(logits, -1)
    nll = logz - jnp.take_along_axis(logits, y[..., None], -1)[..., 0]
    n = jnp.maximum(m.sum(), 1.0)
    return (nll * m).sum() / n, ((logz ** 2) * m).sum() / n

def make_pretrain_loss(mdl, c):
    def loss_fn(p, batch):                                   # batch: [B, T+1]
        x, y = batch[:, :-1], batch[:, 1:]
        logits, mtp, aux = mdl.apply({'params': p}, x, train_mtp=True)
        main, z = ce_terms(logits, y, y != PAD)
        total = main + Z_LOSS * z + aux
        mt = jnp.zeros(())
        for j, lg in enumerate(mtp, 1):                      # head j predicts token t+j+1
            yj = jnp.pad(batch[:, j + 1:], ((0, 0), (0, j)))
            l, _ = ce_terms(lg, yj, yj != PAD); mt = mt + l
        if mtp: total = total + MTP_W * mt / len(mtp)
        return total, (main, aux, mt)
    return loss_fn

def make_fwd(c):                                              # dropless inference forward -> logits
    m = VishwamAI(eval_cfg(c))
    return jax.jit(lambda p, x: m.apply({'params': p}, x)[0])

def make_eval_nll(c):
    m = VishwamAI(eval_cfg(c))
    @jax.jit
    def f(p, batch):
        logits, _, _ = m.apply({'params': p}, batch[:, :-1]); y = batch[:, 1:]
        return ce_terms(logits, y, y != PAD)[0]
    return f

def sample_text(fwd, params, prompt_ids, max_new=120, temperature=0.8, top_p=0.9, seed=0, stop=(), buf_len=256):
    ids = list(prompt_ids); buf = np.zeros((1, buf_len), np.int32); buf[0, :len(ids)] = ids; n = len(ids)
    r = np.random.default_rng(seed)
    for _ in range(min(max_new, buf_len - n)):
        lg = np.asarray(fwd(params, jnp.asarray(buf))[0, n - 1], dtype=np.float64) / max(temperature, 1e-4)
        pr = np.exp(lg - lg.max()); pr /= pr.sum()
        o = np.argsort(-pr); cs = np.cumsum(pr[o]); keep = o[:max(1, int((cs < top_p).sum()) + 1)]
        nxt = int(r.choice(keep, p=pr[keep] / pr[keep].sum()))
        if nxt == EOS or nxt in stop: break
        buf[0, n] = nxt; n += 1
    return buf[0, len(prompt_ids):n].tolist()

# ---------- pre-training stage (also used for the YaRN context-extension stage) ----------
def run_pretrain(tag, c, seq_len, token_budget, lr, warmup, params):
    fin = final_path(tag)
    if os.path.exists(fin):
        print(f'[{tag}] already finished -> loading'); return jax.device_put(load_tree(fin, params), REPL)
    mdl = VishwamAI(c)
    mg = nd * max(1, PER_DEVICE_TOKENS // seq_len)                   # sequences per micro-step (global)
    accum = max(1, GLOBAL_TOKENS_PER_STEP // (mg * seq_len)); tok_step = accum * mg * seq_len
    total = max(2, int(token_budget // tok_step))
    sched = optax.warmup_cosine_decay_schedule(0.0, lr, min(warmup, max(1, total // 10)), total, lr * MIN_LR_FRAC)
    tx = optax.chain(optax.clip_by_global_norm(1.0),
                     optax.adamw(sched, b1=0.9, b2=0.95, eps=1e-8, weight_decay=WD,
                                 mask=lambda p: jax.tree.map(lambda x: x.ndim >= 2, p)))
    loss_fn = make_pretrain_loss(mdl, c)
    @partial(jax.jit, donate_argnums=(0, 1))
    def step_fn(p, o, batch):                                        # batch: [accum, B, T+1]
        def micro(carry, mb):
            g_acc, m_acc = carry
            (_, (main, aux, mt)), g = jax.value_and_grad(loss_fn, has_aux=True)(p, mb)
            return (jax.tree.map(jnp.add, g_acc, g), m_acc + jnp.stack([main, aux, mt])), None
        (g, m), _ = jax.lax.scan(micro, (jax.tree.map(jnp.zeros_like, p), jnp.zeros(3)), batch)
        g = jax.tree.map(lambda x: x / batch.shape[0], g)
        upd, o = tx.update(g, o, p)
        return optax.apply_updates(p, upd), o, m / batch.shape[0], optax.global_norm(g)
    params = jax.device_put(params, REPL); opt = jax.device_put(tx.init(params), REPL); step0 = 0
    ck = ckpts(tag)
    if ck:
        st = load_tree(ck[-1], {'params': params, 'opt': opt, 'step': 0})
        params, opt, step0 = jax.device_put(st['params'], REPL), jax.device_put(st['opt'], REPL), int(st['step'])
        print(f'[{tag}] resumed at step {step0}')
    salt = zlib.crc32(tag.encode()) % 100_000
    def get_batch(step):
        r = np.random.default_rng(SEED * 1_000_003 + salt * 7919 + step)       # deterministic -> exact resume
        b = sample_batch(r, accum * mg, seq_len).reshape(accum, mg, seq_len + 1)
        return jax.device_put(b, ACC_ROW)
    eval_nll, fwd = make_eval_nll(c), make_fwd(c)
    def do_eval():
        r = np.random.default_rng(999); out = {}
        for n in names:
            if len(shards[n]['val']) < seq_len + 2: continue
            out[n] = float(eval_nll(params, jax.device_put(sample_batch(r, mg, seq_len, 'val', only=n), ROW)))
        return out
    print(f'[{tag}] {nd} device(s) | seq {seq_len} | {tok_step/1e3:.0f}k tok/step | {total} steps | rope_scale {c.rope_scale}')
    t0, seen = time.time(), 0
    for step in range(step0, total):
        params, opt, m, gn = step_fn(params, opt, get_batch(step)); seen += tok_step
        if (step + 1) % LOG_EVERY == 0 or step == step0 + 4:
            main, aux, mt = [float(v) for v in m]; tps = seen / (time.time() - t0)
            eta = (total - step - 1) * tok_step / tps / 3600
            print(f'[{tag}] {step+1}/{total} nll {main:.3f} ppl {math.exp(main):.1f} aux {aux:.4f} mtp {mt:.3f} '
                  f'gnorm {float(gn):.2f} {tps/1e3:.1f}k tok/s ETA {eta:.1f}h')
            if not math.isfinite(main): raise RuntimeError('loss diverged — lower LR / check data')
        if (step + 1) % min(EVAL_EVERY, max(5, total // 6)) == 0:
            print('  val nll:', {k: round(v, 3) for k, v in do_eval().items()})
            for pr in ['The capital of India is', 'भारत की राजधानी', 'తెలుగు భాష']:
                print('  >', sp.decode(sample_text(fwd, params, [EOS] + sp.encode(pr), 40)).replace('\n', ' ')[:140])
        if (step + 1) % min(CKPT_EVERY, max(5, total // 10)) == 0:      # short runs checkpoint more often
            save_tree(f'{WORK}/ckpt/{tag}_step_{step+1:07d}.msgpack', {'params': params, 'opt': opt, 'step': step + 1})
            for old in ckpts(tag)[:-2]: os.remove(old)
    save_tree(fin, params)
    for old in ckpts(tag): os.remove(old)
    print(f'[{tag}] finished'); return params

## Load the base model from Drive

In [ ]:
# `params` was initialised in the model cell (same structure as the saved tree); load_tree fills it from Drive.
CFG = cfg_ext if CHECKPOINT == 'ctx' else cfg
params = jax.device_put(load_tree(final_path(CHECKPOINT), params), REPL)
n_params = sum(x.size for x in jax.tree.leaves(params))
print(f'loaded {CHECKPOINT}_final.msgpack | {n_params/1e6:.1f}M parameters (incl. training-only MTP heads) | rope_scale={CFG.rope_scale}')

## Export to safetensors (Hugging Face Qwen3-MoE layout)
MTP heads are not exported.

In [ ]:
!pip -q install safetensors
from safetensors.numpy import save_file
OUT = f'{WORK}/export/vishwamai-{PRESET}'; os.makedirs(OUT, exist_ok=True)
hp = jax.device_get(params)
f32 = lambda a: np.asarray(a, dtype=np.float32)
T_ = {'model.embed_tokens.weight': f32(hp['embed']['embedding']), 'model.norm.weight': f32(hp['norm_f']['w'])}
for i in range(CFG.n_layers):
    b, pre = hp[f'block_{i}'], f'model.layers.{i}.'
    for a, n in (('q', 'q_proj'), ('k', 'k_proj'), ('v', 'v_proj'), ('o', 'o_proj')):
        T_[pre + f'self_attn.{n}.weight'] = f32(b['attn'][a]['kernel']).T.copy()          # flax [in,out] -> torch [out,in]
    T_[pre + 'self_attn.q_norm.weight'] = f32(b['attn']['q_norm']['w'])
    T_[pre + 'self_attn.k_norm.weight'] = f32(b['attn']['k_norm']['w'])
    T_[pre + 'input_layernorm.weight'] = f32(b['n1']['w'])
    T_[pre + 'post_attention_layernorm.weight'] = f32(b['n2']['w'])
    T_[pre + 'mlp.gate.weight'] = f32(b['moe']['router']).T.copy()                         # router [d,E] -> [E,d]
    for e in range(CFG.n_experts):
        T_[pre + f'mlp.experts.{e}.gate_proj.weight'] = f32(b['moe']['wg'][e]).T.copy()
        T_[pre + f'mlp.experts.{e}.up_proj.weight'] = f32(b['moe']['wu'][e]).T.copy()
        T_[pre + f'mlp.experts.{e}.down_proj.weight'] = f32(b['moe']['wd'][e]).T.copy()
save_file(T_, f'{OUT}/model.safetensors', metadata={'format': 'pt'})
hf_cfg = {
  "architectures": ["Qwen3MoeForCausalLM"], "model_type": "qwen3_moe", "vocab_size": CFG.vocab,
  "hidden_size": CFG.d_model, "intermediate_size": CFG.d_ff, "moe_intermediate_size": CFG.moe_ff,
  "num_hidden_layers": CFG.n_layers, "num_attention_heads": CFG.n_heads, "num_key_value_heads": CFG.n_kv_heads,
  "head_dim": CFG.d_model // CFG.n_heads, "num_experts": CFG.n_experts, "num_experts_per_tok": CFG.top_k,
  "norm_topk_prob": True, "decoder_sparse_step": 1, "mlp_only_layers": [], "router_aux_loss_coef": CFG.aux_coef,
  "max_position_embeddings": CFG.max_len, "rms_norm_eps": 1e-6, "rope_theta": CFG.rope_theta,
  "tie_word_embeddings": True, "hidden_act": "silu", "attention_bias": False, "use_sliding_window": False,
  "torch_dtype": "float32", "bos_token_id": BOS, "eos_token_id": EOS, "pad_token_id": PAD}
if CFG.rope_scale > 1:
    hf_cfg["rope_scaling"] = {"rope_type": "yarn", "factor": CFG.rope_scale, "original_max_position_embeddings": CFG.rope_orig_len}
json.dump(hf_cfg, open(f'{OUT}/config.json', 'w'), indent=2)
shutil.copy(SP_PATH, f'{OUT}/tokenizer.model')
json.dump({"tokenizer_class": "LlamaTokenizer", "legacy": False, "model_max_length": CFG.max_len,
           "bos_token": "<s>", "eos_token": "</s>", "unk_token": "<unk>", "pad_token": "<pad>"}, open(f'{OUT}/tokenizer_config.json', 'w'))
print('saved', OUT, f'{os.path.getsize(OUT + "/model.safetensors")/1e6:.0f} MB')

### Verify: exported weights reproduce the JAX model

In [ ]:
!pip -q install -U transformers          # Colab already ships torch; keep its build
import torch
from transformers import AutoModelForCausalLM
hf = AutoModelForCausalLM.from_pretrained(OUT, torch_dtype=torch.float32).eval()
ids = np.array([[EOS] + sp.encode("The capital of India is New Delhi. భారత రాజధాని ఢిల్లీ.")], dtype=np.int32)
with torch.no_grad(): ref = hf(torch.tensor(ids, dtype=torch.long)).logits.numpy()
mine = np.asarray(make_fwd(CFG)(params, jnp.asarray(ids)))
print('max |logit diff| HF vs JAX:', float(np.abs(ref - mine).max()), '| same argmax:', bool((ref.argmax(-1) == mine.argmax(-1)).all()))

## Upload to Hugging Face
Set `LICENSE` yourself (the notebook refuses to upload without it): the pre-training mix includes sources with their own terms, so choose deliberately. The repo is created **private**; flip `PRIVATE = False` when you are happy with the model card.

In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer

LICENSE = None            # e.g. 'apache-2.0' - YOU must choose; see the data sources' terms
PRIVATE = True
REPO_NAME = 'vishwamai-base-v0'

assert LICENSE, "set LICENSE before uploading"
assert os.environ.get('HF_TOKEN'), "add HF_TOKEN as a Colab secret (notebook access on) and re-run the setup cell"
api = HfApi(token=os.environ['HF_TOKEN'])
REPO_ID = f"{api.whoami()['name']}/{REPO_NAME}"

# the exported tokenizer must give the same ids as the SentencePiece model used in training
tk = AutoTokenizer.from_pretrained(OUT)
probe = "Hello, how are you? మీరు ఎలా ఉన్నారు? आप कैसे हैं?"
assert tk.encode(probe, add_special_tokens=False) == sp.encode(probe), 'HF tokenizer ids differ from training tokenizer'
print('tokenizer ids match the training tokenizer')

total_m = n_params / 1e6
card = f"""---
license: {LICENSE}
language: [en, hi, te]
library_name: transformers
tags: [moe, base-model, experimental, from-scratch, jax]
---
# {REPO_NAME}

An **experimental, very early base model**: a small mixture-of-experts decoder (Qwen3-MoE layout) pre-trained from random initialisation with JAX/Flax. It is published so the architecture, tokenizer and export can be inspected and built on, **not** because it is useful yet.

## Model
* {CFG.n_layers} layers, hidden size {CFG.d_model}, {CFG.n_heads} heads / {CFG.n_kv_heads} KV heads, QK-norm, GQA
* {CFG.n_experts} SwiGLU experts per layer, top-{CFG.top_k} routing (~132.7M parameters in total, ~54.9M active per token)
* SentencePiece BPE tokenizer, vocabulary {CFG.vocab} (English, Hindi, Telugu, code), byte-fallback
* Context {CFG.max_len} tokens ({'YaRN-extended from ' + str(CFG.rope_orig_len) if CFG.rope_scale > 1 else 'plain RoPE'})
* Multi-token-prediction heads were used only for training and are not in these weights

## Training
Pre-training on a Colab T4 in float32 with a **~30M-token** budget (a toy-scale run) on English, Hindi and Telugu text plus code, math and security text, followed by a very short long-context extension. **No supervised fine-tuning, no preference tuning, and no distillation or teacher-model outputs** went into this checkpoint. The datasets and their licences are listed in the training notebook; check them before reuse.

## Evaluation (from the pre-training notebook run, 0-shot)
| Benchmark | Score |
|---|---|
| MMLU | ~0.24-0.26 (chance is 0.25) |
| MMMLU-Hindi | ~0.24-0.25 |
| GSM8K | 0.0 |

At this scale the model produces mostly incoherent text. Treat it as a starting checkpoint for further pre-training, not as an assistant.

## Usage
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained("{REPO_ID}")
model = AutoModelForCausalLM.from_pretrained("{REPO_ID}")
```
Requires a `transformers` version with `Qwen3Moe` support.

## Limitations
Not safe for any production use: no safety tuning, near-chance benchmark scores, and training data of mixed provenance.
"""
open(f'{OUT}/README.md', 'w', encoding='utf-8').write(card)

api.create_repo(REPO_ID, repo_type='model', private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO_ID, repo_type='model', commit_message='Upload VishwamAI base checkpoint')
print('uploaded ->', f'https://huggingface.co/{REPO_ID}', '(private)' if PRIVATE else '(public)')